In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("orders.csv")

In [3]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       642 non-null    int64  
 1   order_date     642 non-null    object 
 2   customer_id    642 non-null    object 
 3   country        642 non-null    object 
 4   revenue        642 non-null    int64  
 5   shipping_cost  642 non-null    int64  
 6   refund_amount  642 non-null    float64
dtypes: float64(1), int64(3), object(3)
memory usage: 27.7+ KB


In [5]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       642 non-null    int64         
 1   order_date     642 non-null    datetime64[ns]
 2   customer_id    642 non-null    object        
 3   country        642 non-null    object        
 4   revenue        642 non-null    int64         
 5   shipping_cost  642 non-null    int64         
 6   refund_amount  642 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(2)
memory usage: 30.2+ KB


In [6]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [7]:
orders.describe()

,order_id,order_date,revenue,shipping_cost,refund_amount
count,642.000000,642,642.000000,642.000000,642.000000
mean,1321.500000,2026-02-14 16:24:40.373831680,175.105919,11.954829,9.004673
min,1001.000000,2026-01-01 00:00:00,50.000000,5.000000,0.000000
25%,1161.250000,2026-01-24 00:00:00,113.000000,8.000000,0.000000
50%,1321.500000,2026-02-15 00:00:00,177.500000,12.000000,0.000000
75%,1481.750000,2026-03-08 00:00:00,236.000000,16.000000,12.800000
max,1642.000000,2026-03-31 00:00:00,299.000000,19.000000,59.800000
std,185.473718,NaN,71.792612,4.426500,16.508644


In [8]:
(orders["refund_amount"] > orders["revenue"]).sum()

np.int64(0)

In [9]:
refund_rate = (orders["refund_amount"] > 0).mean()
avg_refund = orders["refund_amount"].mean()

refund_rate, avg_refund

(np.float64(0.26947040498442365), np.float64(9.004672897196262))

Note: High refund rate negatively impacts profitability

In [10]:
orders["order_date"].nunique()

90

In [11]:
orders.groupby("order_date").size().head()

order_date
2026-01-01    8
2026-01-02    7
2026-01-03    6
2026-01-04    5
2026-01-05    5
dtype: int64

In [12]:
orders.to_csv("orders_clean.csv", index=False)

In [37]:
order_items = pd.read_csv("order_items.csv")

In [38]:
missing_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
len(missing_orders)

0

In [39]:
order_items.duplicated().sum()

np.int64(25)

In [40]:
order_items[order_items.duplicated(keep=False)].sort_values(by="order_id")

,order_id,product_id,category,quantity,price
99,1063,P004,Accessories,2,104.5
100,1063,P004,Accessories,2,104.5
177,1115,P004,Accessories,2,148.5
178,1115,P004,Accessories,2,148.5
184,1120,P006,Home,2,108.5
185,1120,P006,Home,2,108.5
202,1133,P001,Electronics,2,95.0
203,1133,P001,Electronics,2,95.0
205,1135,P003,Home,1,38.5
206,1135,P003,Home,1,38.5


In [41]:
order_items = order_items.drop_duplicates()

In [42]:
order_items.duplicated().sum()

np.int64(0)

In [43]:
order_items.to_csv("order_items_clean.csv", index=False)

In [44]:
orders = pd.read_csv("orders_clean.csv")
order_items = pd.read_csv("order_items_clean.csv")

In [45]:
order_items["item_total"] = order_items["quantity"] * order_items["price"]

In [46]:
items_agg = order_items.groupby("order_id")["item_total"].sum().reset_index()

In [48]:
merged = orders.merge(items_agg, on="order_id", how="left")

In [55]:
merged["diff"] = merged["revenue"] - merged["item_total"]

In [56]:
(merged["diff"] != 0).mean()

np.float64(0.5919003115264797)

Note:Product-level data does not fully reconcile with order revenue, indicating inconsistencies in item-level pricing or data structure

In [57]:
merged["final_revenue"] = merged["revenue"]

Note:Item-level data does not fully reconcile with order-level revenue (~59% mismatch), therefore revenue analysis is based on order data, while product analysis is used directionally.

In [58]:
ga4 = pd.read_csv("ga4_traffic.csv")

In [59]:
ga4.info()
ga4.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         270 non-null    object
 1   source       270 non-null    object
 2   medium       270 non-null    object
 3   sessions     270 non-null    int64 
 4   users        270 non-null    int64 
 5   add_to_cart  270 non-null    int64 
 6   purchases    270 non-null    int64 
dtypes: int64(4), object(3)
memory usage: 11.7+ KB


,date,source,medium,sessions,users,add_to_cart,purchases
0,2026-01-01,google,cpc,1284,952,329,47
1,2026-01-01,facebook,paid,1915,1415,543,59
2,2026-01-01,organic,seo,1200,901,288,42
3,2026-01-02,google,cpc,1657,1454,392,61
4,2026-01-02,facebook,paid,1737,1322,368,39


In [60]:
(
    (ga4["sessions"] >= ga4["users"]) &
    (ga4["users"] >= ga4["add_to_cart"]) &
    (ga4["add_to_cart"] >= ga4["purchases"])
).all()

np.True_

In [61]:
ga4["cr_purchase"] = ga4["purchases"] / ga4["sessions"]
ga4["cr_cart"] = ga4["add_to_cart"] / ga4["sessions"]

ga4[["cr_purchase", "cr_cart"]].describe()

,cr_purchase,cr_cart
count,270.000000,270.000000
mean,0.029801,0.224984
std,0.008675,0.041533
min,0.012537,0.150046
25%,0.023516,0.191879
50%,0.029900,0.224001
75%,0.036380,0.259763
max,0.049430,0.299250


In [62]:
ga4_daily = ga4.groupby("date")["purchases"].sum().reset_index()

In [64]:
orders_daily = orders.groupby("order_date")["order_id"].count().reset_index()
orders_daily.columns = ["date", "orders"]

In [66]:
compare = ga4_daily.merge(orders_daily, on="date", how="left")

In [68]:
compare["diff"] = compare["orders"] - compare["purchases"]

In [70]:
compare[["orders", "purchases", "diff"]].describe()

,orders,purchases,diff
count,90.000000,90.000000,90.000000
mean,7.133333,124.366667,-117.233333
std,1.367356,23.592253,23.717935
min,5.000000,74.000000,-179.000000
25%,6.000000,106.500000,-133.000000
50%,7.000000,124.500000,-116.500000
75%,8.000000,140.500000,-100.000000
max,9.000000,184.000000,-65.000000


In [71]:
ga4.to_csv("ga4_traffic_clean.csv", index=False)

In [72]:
ads = pd.read_csv("ads_spend.csv")

In [73]:
ads.info()
ads.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         360 non-null    object
 1   platform     360 non-null    object
 2   campaign     360 non-null    object
 3   spend        360 non-null    int64 
 4   clicks       360 non-null    int64 
 5   impressions  360 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 12.7+ KB


,date,platform,campaign,spend,clicks,impressions
0,2026-01-01,facebook,fb_prospecting,484,1450,36940
1,2026-01-01,facebook,fb_retargeting,335,515,15452
2,2026-01-01,google,google_search,494,1295,25248
3,2026-01-01,google,google_shopping,364,1513,22321
4,2026-01-02,facebook,fb_prospecting,677,1407,34308


In [74]:
ads["cpc"] = ads["spend"] / ads["clicks"]
ads["ctr"] = ads["clicks"] / ads["impressions"]

ads[["cpc", "ctr"]].describe()

,cpc,ctr
count,360.000000,360.000000
mean,0.389074,0.053430
std,0.105069,0.015111
min,0.189908,0.027673
25%,0.307157,0.042071
50%,0.377519,0.051261
75%,0.458502,0.062282
max,0.742000,0.106188


In [75]:
ads_daily = ads.groupby(["date", "platform"])["spend"].sum().reset_index()
ads_daily.columns = ["date", "channel", "spend"]

In [77]:
ads_daily.head()

,date,channel,spend
0,2026-01-01,facebook,819
1,2026-01-01,google,858
2,2026-01-02,facebook,1048
3,2026-01-02,google,868
4,2026-01-03,facebook,955


In [78]:
ga4_daily = ga4.groupby(["date", "source"])[["sessions", "add_to_cart", "purchases"]].sum().reset_index()

ga4_daily.columns = ["date", "channel", "sessions", "add_to_cart", "purchases"]

In [81]:
ga4_daily.head(6)

,date,channel,sessions,add_to_cart,purchases
0,2026-01-01,facebook,1915,543,59
1,2026-01-01,google,1284,329,47
2,2026-01-01,organic,1200,288,42
3,2026-01-02,facebook,1737,368,39
4,2026-01-02,google,1657,392,61
5,2026-01-02,organic,1327,261,35


In [82]:
traffic_share = ga4_daily.copy()
traffic_share["total_sessions"] = traffic_share.groupby("date")["sessions"].transform("sum")
traffic_share["share"] = traffic_share["sessions"] / traffic_share["total_sessions"]
traffic_share.head()

,date,channel,sessions,add_to_cart,purchases,total_sessions,share
0,2026-01-01,facebook,1915,543,59,4399,0.435326
1,2026-01-01,google,1284,329,47,4399,0.291885
2,2026-01-01,organic,1200,288,42,4399,0.272789
3,2026-01-02,facebook,1737,368,39,4721,0.367931
4,2026-01-02,google,1657,392,61,4721,0.350985


In [83]:
orders_daily = orders.groupby("order_date")[["revenue", "refund_amount"]].sum().reset_index()
orders_daily.columns = ["date", "revenue", "refund"]

In [84]:
orders_allocated = traffic_share.merge(orders_daily, on="date", how="left")
orders_allocated["revenue"] = orders_allocated["revenue"] * orders_allocated["share"]
orders_allocated["refund"] = orders_allocated["refund"] * orders_allocated["share"]
orders_allocated.head()

,date,channel,sessions,add_to_cart,purchases,total_sessions,share,revenue,refund
0,2026-01-01,facebook,1915,543,59,4399,0.435326,540.675153,41.791316
1,2026-01-01,google,1284,329,47,4399,0.291885,362.520573,28.020914
2,2026-01-01,organic,1200,288,42,4399,0.272789,338.804274,26.187770
3,2026-01-02,facebook,1737,368,39,4721,0.367931,541.593730,56.882059
4,2026-01-02,google,1657,392,61,4721,0.350985,516.649862,54.262275


In [87]:
final_table = orders_allocated.merge(
    ga4_daily[["date", "channel", "sessions", "add_to_cart", "purchases"]],
    on=["date", "channel"],
    how="left"
).merge(
    ads_daily,
    on=["date", "channel"],
    how="left"
)
final_table["spend"] = final_table["spend"].fillna(0)
final_table.head()

,date,channel,sessions_x,add_to_cart_x,purchases_x,total_sessions,share,revenue,refund,sessions_y,add_to_cart_y,purchases_y,spend
0,2026-01-01,facebook,1915,543,59,4399,0.435326,540.675153,41.791316,1915,543,59,819.0
1,2026-01-01,google,1284,329,47,4399,0.291885,362.520573,28.020914,1284,329,47,858.0
2,2026-01-01,organic,1200,288,42,4399,0.272789,338.804274,26.187770,1200,288,42,0.0
3,2026-01-02,facebook,1737,368,39,4721,0.367931,541.593730,56.882059,1737,368,39,1048.0
4,2026-01-02,google,1657,392,61,4721,0.350985,516.649862,54.262275,1657,392,61,868.0


In [88]:
final_table["profit"] = final_table["revenue"] - final_table["refund"] - final_table["spend"]
final_table["roas"] = final_table["revenue"] / final_table["spend"]
final_table["conversion_rate"] = final_table["purchases"] / final_table["sessions"]

<class 'KeyError'>: 'purchases'